# Pydantic Basics: Creating and Validating Models

Welcome to this tutorial on **Pydantic**, the most widely used data validation library for Python! 

Pydantic guarantees data correctness at runtime by using Python's native type annotations. It parses input data into strongly-typed objects and raises friendly, descriptive validation errors when constraints are violated.

In this notebook, you will learn the fundamentals of:
1. Installing Pydantic and importing essential components.
2. Defining data schemas by inheriting from `BaseModel`.
3. Using `Field` and `Annotated` to add validation constraints, titles, and documentation to fields.
4. Simulating input data and handling parsing/validation errors (`ValidationError`).


In [1]:
!uv pip install pydantic --quiet


## 1. Imports

We import:
*   `BaseModel`: The foundation class for all Pydantic schemas.
*   `Field`: Used to define constraints (e.g., numeric limits, string length), metadata, and custom default values.
*   `Annotated`: A standard Python typing module (from PEP 593) that allows attaching metadata (like Pydantic's `Field` properties) to existing type annotations.
*   Built-in Pydantic validated types like `EmailStr` and `AnyUrl` for automated string format checks.


In [2]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field
from typing import List, Dict, Optional, Annotated


## 2. Defining a Pydantic Model

A model is declared by creating a subclass of `BaseModel`. Each field must have a type annotation. Pydantic leverages these annotations to cast/validate incoming values.

In the `Patient` model:
*   `name`: Declared using `Annotated[str, Field(...)]`. It has a maximum length of 150 characters, title, description, and example metadata.
*   `age`: Standard syntax using `Field(gt=0, lt=120)`. The age must be strictly greater than 0 and less than 120.
*   `linkedin_url`: An optional field of type `AnyUrl` defaulting to `None`.
*   `weight`: Uses `strict=True`. In strict mode, Pydantic will fail if the input is not a float (e.g., a string `'67.9'` will raise a validation error instead of being coerced).
*   `married`: A boolean field defaulting to `None`.
*   `allergies`: An optional list of strings with a maximum of 5 items.
*   `contact_details`: A dictionary mapping string keys to string values.


In [3]:
class Patient(BaseModel):

    # Using Annotated allows us to cleanly specify the type first, and attach metadata inside Field()
    name: str = Annotated[str, Field(max_length=150, title='Name of the patient', description='Patient Name for records', examples=['John Doe'])]
    
    # Numeric boundary validators: age must be > 0 and < 120
    age: int = Field(gt=0, lt=120)
    
    # Optional field: defaults to None if not provided in input data
    linkedin_url: Optional[AnyUrl] = None
    
    # strict=True disables type coercion (e.g. passing a string '65.5' instead of float 65.5 will raise an error)
    weight: Annotated[float, Field(gt=0, description='Submit patient weight for the report', strict=True)]
    
    # Default boolean configuration
    married: Annotated[bool, Field(default=None, description='Is the patient married or not')]
    
    # Collection constraints: list of strings, maximum 5 items allowed
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=5)]
    
    # Arbitrary key-value dictionary field
    contact_details: Dict[str, str]


## 3. Sub-Models and Nesting Preview

In early development stages, nested data is often structured as standard dictionaries. However, nesting models offers much cleaner type-safety. Below is a commented-out preview of how we could split contact details into its own validated model.


In [5]:
# class ContactDetails(BaseModel):
#     email: EmailStr
#     cell_number: str


## 4. Downstream Data Pipelines (Mock Functions)

These helper functions represent business logic (like writing data to a database or updating records) that executes only after Pydantic guarantees data validity.


In [6]:
def insert_patient_data(patient: Patient):
    # This function safely assumes 'patient' has correct types and valid properties
    print(f"Patient Name: {patient.name}")
    print(f"Patient Age: {patient.age}")
    print(f"Patient Weight: {patient.weight}")
    print(f"Patient Married Status: {patient.married}")
    print(f"Patient Allergies: {patient.allergies}")
    print(f"Patient Contact Details: {patient.contact_details}")
    print('Patient info inserted')


In [7]:
def update_patient_data(patient: Patient):
    # Simulates updating an existing patient record
    print(f"Patient Name: {patient.name}")
    print(f"Patient Age: {patient.age}")
    print(f"Patient Weight: {patient.weight}")
    print(f"Patient Married Status: {patient.married}")
    print(f"Patient Allergies: {patient.allergies}")
    print(f"Patient Contact Details: {patient.contact_details}")
    print('Patient info updateed')


## 5. Instantiation and ValidationError Handling

Let's attempt to load a patient record. Note that the weight is set to `-67.9` (violating `gt=0`) and the contact details are defined as a dictionary.


In [15]:
patient_info = {
    'name': 'Kevin',
    'age': "26",
    'weight': -67.9, # Invalid: weight must be > 0
    'married': False,
    'allergies': ['lactose', 'dust'],
    'contact_details': {
        'email': 'example@domain.io',
        'cell_number': '9999999999'
    }
}


Running the cell below will raise a `ValidationError`. Pydantic returns structured error messages showing exactly:
1. Which field failed (`weight`).
2. Why it failed (`Input should be greater than 0`).
3. The type of failure (`type=greater_than`).


In [18]:
try:
    patient = Patient(**patient_info)
except Exception as e:
    print("Pydantic caught an invalid state!")
    print(e)


Pydantic caught an invalid state!
1 validation error for Patient
weight
  Input should be greater than 0 [type=greater_than, input_value=-67.9, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than


## 6. Correcting Data and Executing Pipelines

Now, we correct the data by setting weight to `67.9` (a valid float > 0), instantiate the model successfully, and send it to our downstream processing functions.


In [19]:
# Correcting the invalid weight field in patient_info
patient_info['weight'] = 67.9

# Validation succeeds!
patient = Patient(**patient_info)
print("Patient instantiated successfully:")
print(patient)


Patient instantiated successfully:
name='Kevin' age=26 linkedin_url=None weight=67.9 married=False allergies=['lactose', 'dust'] contact_details={'email': 'example@domain.io', 'cell_number': '9999999999'}


In [11]:
insert_patient_data(patient=patient)


Patient Name: Kevin
Patient Age: 26
Patient Weight: 67.9
Patient Married Status: False
Patient Allergies: ['lactose', 'dust']
Patient Contact Details: {'email': 'example@domain.io', 'cell_number': '9999999999'}
Patient info inserted


In [12]:
update_patient_data(patient=patient)


Patient Name: Kevin
Patient Age: 26
Patient Weight: 67.9
Patient Married Status: False
Patient Allergies: ['lactose', 'dust']
Patient Contact Details: {'email': 'example@domain.io', 'cell_number': '9999999999'}
Patient info updateed
